# Week 4: Data Quality Checks (UTM FAI Student FAQ Assistant)

Runs the two validation rules and supporting checks from `docs/week4_data_quality_audit.md`
on the prepared chunks, and writes the results to `data/quality_results.json`.

**Run from the repository root** (the folder that contains `data/`). Python 3 standard library only.
The last section (PDF checks) is optional: it needs `pypdf` and the two original PDFs in `data/raw/`
(they are not committed). If they are missing, that section is skipped and says so.

| Rule | Problem it detects | Condition |
|---|---|---|
| Rule 1: readable text | Extraction damage (font-glyph codes instead of words) | Chunk text contains `/gid` followed by 5 digits |
| Rule 2: page reference present | Chunk cannot be cited to a page (R2, FR3) | `page` is not empty |

In [1]:
!git clone https://github.com/addyyadiyy/ai-engineering-utm-fai-student-faq-assistant.git
%cd ai-engineering-utm-fai-student-faq-assistant

Cloning into 'ai-engineering-utm-fai-student-faq-assistant'...
remote: Enumerating objects: 291, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 291 (delta 41), reused 0 (delta 0), pack-reused 209 (from 1)
Receiving objects: 100% (291/291), 1.82 MiB | 11.12 MiB/s, done.
Resolving deltas: 100% (123/123), done.
/content/ai-engineering-utm-fai-student-faq-assistant


In [2]:
import json, re, hashlib, collections
from pathlib import Path
from datetime import datetime, timezone

# --- find the repository root (folder that contains data/processed) ---
ROOT = Path.cwd()
for cand in [ROOT, *ROOT.parents]:
    if (cand / "data" / "processed").is_dir():
        ROOT = cand
        break
else:
    raise FileNotFoundError("Run this notebook from the repository root (needs data/processed/).")

SOURCES = {
    "S01": {"processed": "data/processed/prepared_text_S01.jsonl",
            "pipeline":  "data/pipeline_v1/documents_S01_regulations/prepared_text.jsonl",
            "pdf": "data/raw/FT-UG-ACADEMIC-REGULATIONS-16th-edition.pdf"},
    "S02": {"processed": "data/processed/prepared_text_S02.jsonl",
            "pipeline":  "data/pipeline_v1/documents_S02_programme/prepared_text.jsonl",
            "pdf": "data/raw/Bachelor-in-Artificial-Intelligence-With-Honours-FAI-programme-page.pdf"},
}

def load_jsonl(rel):
    with open(ROOT / rel, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

chunks   = {s: load_jsonl(v["processed"]) for s, v in SOURCES.items()}
pipeline = {s: load_jsonl(v["pipeline"])  for s, v in SOURCES.items()}
results  = {"generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "pipeline_version": "week3-v1", "rules": {}, "supporting_checks": {}, "pdf_checks": {}}

print("Repository root:", ROOT)
for s in SOURCES:
    print(f"{s}: {len(chunks[s])} chunks in data/processed, {len(pipeline[s])} in pipeline_v1")
print("Total chunks:", sum(len(v) for v in chunks.values()))

Repository root: /content/ai-engineering-utm-fai-student-faq-assistant
S01: 213 chunks in data/processed, 213 in pipeline_v1
S02: 57 chunks in data/processed, 57 in pipeline_v1
Total chunks: 270


## Rule 1: readable text (glyph-code check)

In [3]:
GLYPH = re.compile(r"/gid\d{5}")

rule1 = {"problem": "Extraction damage: font-glyph codes instead of words",
         "condition": "chunk text contains '/gid' followed by 5 digits",
         "action": "Send failed chunks to a person for review; rebuild by hand or exclude",
         "per_source": {}}
for s, recs in chunks.items():
    failed = [i + 1 for i, r in enumerate(recs) if GLYPH.search(r["text"])]
    share = None
    if failed:
        glyph_chars = sum(len("".join(GLYPH.findall(recs[i - 1]["text"]))) for i in failed)
        total_chars = sum(len(recs[i - 1]["text"]) for i in failed)
        share = round(100 * glyph_chars / total_chars, 1)
    rule1["per_source"][s] = {"chunks": len(recs), "failed": len(failed), "failed_chunk_numbers": failed,
                              "glyph_share_of_failed_chars_pct": share}
    print(f"{s}: {len(failed)} of {len(recs)} chunks FAIL Rule 1", failed, f"(glyph codes = {share}% of their characters)" if share else "")
results["rules"]["rule1_readable_text"] = rule1

for i in rule1["per_source"]["S01"]["failed_chunk_numbers"][:1]:
    print("\nExample failing chunk #%d (first 160 chars):" % i)
    print(repr(chunks["S01"][i - 1]["text"][:160]))

S01: 2 of 213 chunks FAIL Rule 1 [1, 2] (glyph codes = 94.6% of their characters)
S02: 0 of 57 chunks FAIL Rule 1 [] 

Example failing chunk #1 (first 160 chars):
'www.penerbit.utm.my\n2026\n/gid00006/gid00031/gid00036/gid00047/gid00042/gid00045/gid00046\n/gid00014/gid00042/gid00035/gid00031/gid00001/gid00020/gid00035/gid0002'


## Rule 2: page reference present

In [4]:
REQUIRED = ["record_id", "text", "source", "source_sha256", "page"]

rule2 = {"problem": "Chunk cannot be cited to a source page (R2, FR3)",
         "condition": "'page' is not empty (also checks record_id, text, source, source_sha256)",
         "action": "Re-extract page by page so each chunk stores its page, then re-run",
         "per_source": {}}
for s, recs in chunks.items():
    missing = {k: sum(1 for r in recs if r.get(k) in (None, "")) for k in REQUIRED}
    rule2["per_source"][s] = {"chunks": len(recs), "failed_page_empty": missing["page"], "missing_by_field": missing}
    print(f"{s}: page empty in {missing['page']} of {len(recs)} chunks | other missing fields: " +
          str({k: v for k, v in missing.items() if k != 'page'}))
total_fail = sum(v["failed_page_empty"] for v in rule2["per_source"].values())
print("\nRule 2 total failures:", total_fail, "of", sum(len(v) for v in chunks.values()))
results["rules"]["rule2_page_reference"] = rule2

# The pipeline_v1 output uses a different record format
for s, recs in pipeline.items():
    print(f"{s} pipeline_v1 fields: {sorted(recs[0].keys())}")

S01: page empty in 213 of 213 chunks | other missing fields: {'record_id': 0, 'text': 0, 'source': 0, 'source_sha256': 0}
S02: page empty in 57 of 57 chunks | other missing fields: {'record_id': 0, 'text': 0, 'source': 0, 'source_sha256': 0}

Rule 2 total failures: 270 of 270
S01 pipeline_v1 fields: ['id', 'page', 'source', 'text']
S02 pipeline_v1 fields: ['id', 'page', 'source', 'text']


## Supporting checks (all chunks)

In [5]:
BOILERPLATE = re.compile(r"Copyright ©|All Right Reserved|Hak cipta terpelihara|penerbit\.utm")
norm = lambda t: re.sub(r"\W+", " ", t.lower()).strip()

sup = {}
for s, recs in chunks.items():
    texts = [r["text"] for r in recs]
    counts = collections.Counter(norm(t) for t in texts)
    sup[s] = {
        "chunks": len(recs),
        "empty_text": sum(1 for t in texts if not t.strip()),
        "duplicate_groups": sum(1 for c in counts.values() if c > 1),
        "under_50_chars": [i + 1 for i, t in enumerate(texts) if len(t.strip()) < 50],
        "boilerplate_chunks": [i + 1 for i, t in enumerate(texts) if BOILERPLATE.search(t)],
        "start_lowercase": sum(1 for t in texts if t.lstrip()[:1].islower()),
        "start_lowercase_pct": round(100 * sum(1 for t in texts if t.lstrip()[:1].islower()) / len(texts), 1),
        "processed_vs_pipeline_text_identical": all(a["text"] == b["text"] for a, b in zip(recs, pipeline[s])) and len(recs) == len(pipeline[s]),
    }
    print(s, json.dumps(sup[s]))
results["supporting_checks"] = sup

S01 {"chunks": 213, "empty_text": 0, "duplicate_groups": 0, "under_50_chars": [], "boilerplate_chunks": [1, 4], "start_lowercase": 175, "start_lowercase_pct": 82.2, "processed_vs_pipeline_text_identical": true}
S02 {"chunks": 57, "empty_text": 0, "duplicate_groups": 0, "under_50_chars": [6, 30, 42, 43], "boilerplate_chunks": [1, 7, 10, 40, 49, 52, 56], "start_lowercase": 20, "start_lowercase_pct": 35.1, "processed_vs_pipeline_text_identical": true}


## PDF checks (optional)

Needs `pypdf` and the original PDFs in `data/raw/`. Confirms the PDFs are the files in the manifest,
lists pages with no text, and finds which PDF page each chunk came from.

In [6]:
try:
    import pypdf
except ImportError:
    pypdf = None

manifest = json.load(open(ROOT / "data" / "week3_manifest.json", encoding="utf-8"))
expected_sha = {r["source_id"]: r["source_sha256"] for r in manifest["records"]}

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

squash = lambda t: re.sub(r"\s+", "", t)

if pypdf is None:
    print("pypdf is not installed: PDF checks skipped (pip install pypdf).")
    results["pdf_checks"] = {"status": "skipped: pypdf not installed"}
else:
    for s, v in SOURCES.items():
        path = ROOT / v["pdf"]
        if not path.exists():
            print(f"{s}: {v['pdf']} not found: PDF checks skipped for this source.")
            results["pdf_checks"][s] = {"status": "skipped: PDF not found in data/raw/"}
            continue
        reader = pypdf.PdfReader(str(path))
        pages = [(p.extract_text() or "") for p in reader.pages]
        sq_pages = [squash(p) for p in pages]
        joined = squash("\n".join(pages))
        empty_pages = [i + 1 for i, p in enumerate(pages) if len(p.strip()) < 20]

        single, multi, notfound = 0, 0, []
        for i, r in enumerate(chunks[s]):
            t = squash(r["text"])
            if any(t in p for p in sq_pages):
                single += 1
            elif t in joined:
                multi += 1
            else:
                notfound.append(i + 1)
        info = {"status": "ok", "pdf": v["pdf"], "pages": len(pages),
                "sha256_matches_manifest": sha256(path) == expected_sha.get(s),
                "pages_with_no_text": empty_pages,
                "chunks_found_on_one_page": single,
                "chunks_crossing_a_page_break": multi,
                "chunks_not_found_in_pdf_text": notfound}
        results["pdf_checks"][s] = info
        print(s, json.dumps(info))

pypdf is not installed: PDF checks skipped (pip install pypdf).


## Save results

In [7]:
out = ROOT / "data" / "quality_results.json"
with open(out, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("Saved", out.relative_to(ROOT))

r1 = results["rules"]["rule1_readable_text"]["per_source"]
r2 = results["rules"]["rule2_page_reference"]["per_source"]
print()
print("SUMMARY")
print(f"  Rule 1 (readable text):   S01 {r1['S01']['failed']}/{r1['S01']['chunks']} fail, S02 {r1['S02']['failed']}/{r1['S02']['chunks']} fail")
print(f"  Rule 2 (page reference):  S01 {r2['S01']['failed_page_empty']}/{r2['S01']['chunks']} fail, S02 {r2['S02']['failed_page_empty']}/{r2['S02']['chunks']} fail")

Saved data/quality_results.json

SUMMARY
  Rule 1 (readable text):   S01 2/213 fail, S02 0/57 fail
  Rule 2 (page reference):  S01 213/213 fail, S02 57/57 fail
